# 06 · Advanced asyncio: How It Works, Coordination, Cancellation, Context & Debugging

**Module 13 – Concurrency & asyncio** · Estimated time: 80 min

> Assumes lesson 05 (coroutines, `gather`, `TaskGroup`, timeouts, queues). Here is everything else a professional async developer needs.

### 🎯 What you will learn
1. How it really works: the **event loop**, coroutines, `Task`s, `Future`s, the awaitable protocol (`__await__`)
2. Running things: `asyncio.run`, `asyncio.Runner`, background tasks (and the "task disappeared" bug)
3. Waiting: `gather(return_exceptions=)`, `wait`, `as_completed` (async iteration), `TaskGroup` vs `gather`
4. **Cancellation semantics** in depth: `CancelledError`, `shield`, cleanup, `timeout_at`, rescheduling
5. Synchronisation: `Lock`, `Event`, `Condition`, `Semaphore`, `Barrier`; queues incl. `PriorityQueue` & `shutdown()`
6. **`contextvars`**: request-scoped data across tasks
7. Async generators & clean-up (`aclose`, `contextlib.aclosing`), async context managers (`asynccontextmanager`)
8. Performance: eager tasks (3.12), uvloop
9. Graceful shutdown with signals
10. **Debugging**: debug mode, slow callbacks, never-awaited coroutines, task introspection, 🆕 call graphs & `python -m asyncio ps/pstree` (3.14)
11. The ecosystem: anyio, trio, httpx, aiofiles, asyncpg

In [1]:
import asyncio, time, contextvars, random

## 1. How asyncio works

```
         ┌──────────────── event loop (one thread) ────────────────┐
ready ──►│ run task A until it awaits something not ready → suspend │
queue    │ run task B ...                                           │
         │ check I/O (selectors / OS), timers → wake waiting tasks │
         └──────────────────────────────────────────────────────────┘
```
- A **coroutine** (from an `async def` call) is a paused function; `await` passes control back to the loop
- A **Task** wraps a coroutine and schedules it on the loop (`asyncio.create_task`)
- A **Future** is a low-level placeholder for a result set later (Tasks are Futures)
- An object is **awaitable** if it's a coroutine, Task, Future, or defines `__await__`

In [2]:
async def compute():
    await asyncio.sleep(0)            # yield to the loop once
    return 42

coro = compute()
print(type(coro).__name__, asyncio.iscoroutine(coro))
task = asyncio.create_task(coro)       # scheduled now (top-level await works in Jupyter's running loop)
print(type(task).__name__, task.done())
print(await task, task.done(), task.result())

coroutine True
Task False
42 True 42


### A low-level `Future` and a custom awaitable

In [3]:
loop = asyncio.get_running_loop()
fut = loop.create_future()
loop.call_later(0.1, fut.set_result, "set by a timer callback")
print(await fut)

class Countdown:
    """Any object with __await__ can be awaited."""
    def __init__(self, n): self.n = n
    def __await__(self):
        for i in range(self.n, 0, -1):
            print("  tick", i)
            yield from asyncio.sleep(0.05).__await__()
        return "liftoff"

print(await Countdown(3))

set by a timer callback
  tick 3
  tick 2


  tick 1
liftoff


## 2. Running asyncio programs

```python
asyncio.run(main())                       # the standard entry point: new loop, run, clean up

with asyncio.Runner(debug=True) as runner:   # 3.11+: several top-level calls on ONE loop
    runner.run(setup())
    runner.run(main())
```
In Jupyter the loop is already running, so use top-level `await` (as in this notebook).

### ❌ The disappearing background task

The loop keeps only **weak** references to tasks. If you don't keep a reference, a "fire-and-forget" task can be garbage-collected mid-run:

In [4]:
background_tasks = set()

async def send_email(to):
    await asyncio.sleep(0.1)
    print(f"  email sent to {to}")

for user in ["a@x.com", "b@x.com"]:
    t = asyncio.create_task(send_email(user), name=f"email-{user}")
    background_tasks.add(t)                       # ✅ keep a strong reference...
    t.add_done_callback(background_tasks.discard)  # ...and drop it when done
await asyncio.sleep(0.2)
print("pending background tasks:", len(background_tasks))

  email sent to a@x.com
  email sent to b@x.com


pending background tasks: 0


✅ Better still: start them inside a `TaskGroup`, so they can't be lost and errors aren't silently swallowed.

## 3. Waiting strategies

In [5]:
async def job(name, delay, fail=False):
    await asyncio.sleep(delay)
    if fail:
        raise ValueError(f"{name} failed")
    return name

# gather: results in order; by default the FIRST exception propagates (other tasks keep running!)
results = await asyncio.gather(job("a", 0.1), job("b", 0.05, fail=True), job("c", 0.15), return_exceptions=True)
print("gather(return_exceptions=True):", results)

# as_completed: process results as they finish (3.13+: async iteration yields the original tasks)
tasks = [asyncio.create_task(job(n, d)) for n, d in [("slow", 0.3), ("fast", 0.1), ("mid", 0.2)]]
async for done_task in asyncio.as_completed(tasks):
    print("finished:", await done_task)

# wait: sets of done/pending with FIRST_COMPLETED / FIRST_EXCEPTION / ALL_COMPLETED
tasks = [asyncio.create_task(job(n, d)) for n, d in [("x", 0.2), ("y", 0.05)]]
done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
print("first:", [t.result() for t in done], "| pending:", len(pending))
for t in pending:
    t.cancel()

gather(return_exceptions=True): ['a', ValueError('b failed'), 'c']


finished: fast


finished: mid
finished: slow


first: ['y'] | pending: 1


| Tool | Use when |
|---|---|
| `TaskGroup` ✅ | a set of related tasks that should **all succeed**; any failure cancels the rest; errors raised as `ExceptionGroup` |
| `gather(..., return_exceptions=True)` | you want every result **including failures**, no cancellation |
| `as_completed` | handle each result ASAP |
| `wait` | fine-grained control (first done, first exception, timeouts) |

## 4. Cancellation in depth

`task.cancel()` throws `CancelledError` **into** the coroutine at its current `await`. Rules:
1. Use `try/finally` (or `async with`) for clean-up
2. If you catch `CancelledError`, **re-raise** it (otherwise the task looks successful and structured concurrency breaks)
3. Never use a bare `except:` / `except BaseException` that swallows it

In [6]:
async def worker(log):
    try:
        while True:
            await asyncio.sleep(0.05)
            log.append("work")
    except asyncio.CancelledError:
        log.append("cleanup")
        raise                                   # ✅ propagate
    finally:
        log.append("finally")

log = []
t = asyncio.create_task(worker(log))
await asyncio.sleep(0.12)
t.cancel()
try:
    await t
except asyncio.CancelledError:
    print("task cancelled:", t.cancelled(), log)

task cancelled: True ['work', 'work', 'cleanup', 'finally']


### `shield`: protect critical work from outer cancellation

In [7]:
async def save_payment():
    await asyncio.sleep(0.2)
    return "payment saved"

async def handle_request():
    critical = asyncio.create_task(save_payment())
    try:
        return await asyncio.shield(critical)   # if we're cancelled, `critical` keeps running
    except asyncio.CancelledError:
        print("request cancelled, but the payment write continues")
        raise

req = asyncio.create_task(handle_request())
await asyncio.sleep(0.05)
req.cancel()
await asyncio.sleep(0.3)
print("request cancelled:", req.cancelled())

request cancelled, but the payment write continues


request cancelled: True


### Timeouts: `timeout`, `timeout_at`, and rescheduling

In [8]:
async def fetch_with_deadline():
    loop = asyncio.get_running_loop()
    async with asyncio.timeout_at(loop.time() + 0.3) as cm:    # absolute deadline
        await asyncio.sleep(0.1)
        cm.reschedule(loop.time() + 0.5)                       # extend it while running
        await asyncio.sleep(0.3)
        return "finished within the extended deadline"

print(await fetch_with_deadline())

finished within the extended deadline


## 5. Synchronisation primitives (single-threaded, but tasks still interleave!)

Race conditions also happen in asyncio: any `await` between a read and a write lets another task run.

In [9]:
balance = 100
lock = asyncio.Lock()

async def withdraw(amount, use_lock):
    global balance
    async def critical():
        global balance
        current = balance
        await asyncio.sleep(0.01)              # e.g. a DB round-trip
        if current >= amount:
            balance = current - amount
            return True
        return False
    if use_lock:
        async with lock:
            return await critical()
    return await critical()

for use_lock in (False, True):
    balance = 100
    ok = await asyncio.gather(*(withdraw(80, use_lock) for _ in range(2)))
    print(f"lock={use_lock}: withdrawals succeeded {ok}, final balance {balance}")

lock=False: withdrawals succeeded [True, True], final balance 20
lock=True: withdrawals succeeded [True, False], final balance 20


Without the lock, **both** ₹80 withdrawals succeed from a ₹100 balance (₹160 paid out!), because both tasks read `100` before either wrote. With the lock, the second withdrawal correctly fails.

In [10]:
# Event, Condition, Semaphore, Barrier
ready = asyncio.Event()
async def waiter(i):
    await ready.wait()
    return f"waiter {i} released"
waiters = [asyncio.create_task(waiter(i)) for i in range(2)]
await asyncio.sleep(0.05); ready.set()
print(await asyncio.gather(*waiters))

barrier = asyncio.Barrier(3)                    # 3.11+
async def phase(name):
    await asyncio.sleep(random.random() / 10)
    await barrier.wait()
    return f"{name} passed the barrier"
print(await asyncio.gather(*(phase(n) for n in "ABC")))

['waiter 0 released', 'waiter 1 released']
['A passed the barrier', 'B passed the barrier', 'C passed the barrier']


### Queues

`asyncio.Queue`, `PriorityQueue`, `LifoQueue`; 🆕 3.13 `queue.shutdown()` makes waiting `get()`s raise `QueueShutDown`:

In [11]:
async def consumer(q, name, out):
    while True:
        try:
            prio, item = await q.get()
        except asyncio.QueueShutDown:
            return
        out.append(f"{name}:{item}")
        q.task_done()

q, out = asyncio.PriorityQueue(), []
for prio, item in [(3, "low"), (1, "urgent"), (2, "normal")]:
    q.put_nowait((prio, item))
consumers = [asyncio.create_task(consumer(q, f"c{i}", out)) for i in range(2)]
await q.join()
q.shutdown()
await asyncio.gather(*consumers)
print(out)

['c0:urgent', 'c0:normal', 'c0:low']


## 6. `contextvars`: request-scoped data

Each task gets a **copy of the context** when it's created, so values set inside one request don't leak into others (thread-locals don't work for asyncio).

In [12]:
request_id = contextvars.ContextVar("request_id", default="-")

def log(msg):
    print(f"[req={request_id.get()}] {msg}")       # no need to pass request_id everywhere

async def handle(rid):
    request_id.set(rid)
    await asyncio.sleep(random.random() / 20)
    log("loaded user")
    await asyncio.sleep(random.random() / 20)
    log("done")

await asyncio.gather(handle("A1"), handle("B2"), handle("C3"))
log("outside any request")

[req=C3] loaded user
[req=B2] loaded user


[req=B2] done


[req=A1] loaded user
[req=C3] done
[req=A1] done
[req=-] outside any request


Web frameworks and logging libraries use this for request IDs, auth info and DB sessions.

## 7. Async generators and async context managers

In [13]:
from contextlib import asynccontextmanager, aclosing

@asynccontextmanager
async def db_connection(name):
    print(f"  open {name}")
    try:
        yield {"name": name}
    finally:
        await asyncio.sleep(0)
        print(f"  close {name}")

async def stream_rows():
    async with db_connection("db") as conn:
        for i in range(100):
            yield f"row {i} from {conn['name']}"

async with aclosing(stream_rows()) as rows:      # guarantees the generator's clean-up runs
    async for row in rows:
        print(row)
        if row.endswith("2 from db"):
            break                                 # stop early → aclose() runs the finally block

  open db
row 0 from db
row 1 from db
row 2 from db
  close db


## 8. Performance

### Eager tasks (3.12+)
Normally `create_task` schedules the coroutine for *later*. With the **eager task factory**, a task starts running immediately and, if it finishes without suspending (e.g. a cache hit), never touches the scheduler:

In [14]:
cache = {"k": "cached"}
async def get(key):
    if key in cache:
        return cache[key]           # returns without awaiting
    await asyncio.sleep(0.01)
    return "fresh"

loop = asyncio.get_running_loop()
loop.set_task_factory(asyncio.eager_task_factory)
t = asyncio.create_task(get("k"))
print("done immediately with an eager factory:", t.done(), t.result())
loop.set_task_factory(None)

done immediately with an eager factory: True cached


### uvloop
`pip install uvloop`: a drop-in, faster event loop (libuv) for servers: `uvloop.run(main())`. Used by uvicorn when installed.

## 9. Graceful shutdown with signals (scripts/servers)

```python
import asyncio, signal

async def main():
    stop = asyncio.Event()
    loop = asyncio.get_running_loop()
    for sig in (signal.SIGINT, signal.SIGTERM):          # Ctrl+C / docker stop / systemd
        loop.add_signal_handler(sig, stop.set)            # (POSIX only)
    async with asyncio.TaskGroup() as tg:
        tg.create_task(serve_forever(stop))
        await stop.wait()                                 # then let tasks finish their current work
        print("shutting down...")

asyncio.run(main())
```
`asyncio.run()` cancels remaining tasks and closes async generators on exit.

## 10. Debugging asyncio

### Debug mode
`asyncio.run(main(), debug=True)` or `PYTHONASYNCIODEBUG=1` reports **never-awaited coroutines**, callbacks that block the loop longer than `loop.slow_callback_duration` (default 0.1 s), and non-thread-safe calls.

In [15]:
import warnings
async def forgotten():
    return 1

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    forgotten()                       # ❌ forgot `await`: the coroutine never runs
    import gc; gc.collect()
print(caught[0].message if caught else "no warning captured")

coroutine 'forgotten' was never awaited


### Introspecting tasks

In [16]:
async def child():
    await asyncio.sleep(0.2)

async def parent():
    async with asyncio.TaskGroup() as tg:
        tg.create_task(child(), name="child-task")
        await asyncio.sleep(0.05)
        print("current task:", asyncio.current_task().get_name())
        print("all tasks   :", sorted(t.get_name() for t in asyncio.all_tasks() if t.get_name() in {"parent-task", "child-task"}))
        print("call graph of the child (3.14+):")
        child_task = next(t for t in asyncio.all_tasks() if t.get_name() == "child-task")
        print(asyncio.format_call_graph(child_task))

await asyncio.create_task(parent(), name="parent-task")

current task: parent-task
all tasks   : ['child-task', 'parent-task']
call graph of the child (3.14+):
* Task(name='child-task', id=0x10bc1d2b0)
  + Call stack:
  |   File '/private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/uvpy/cpython-3.15.0-macos-aarch64-none/lib/python3.15/asyncio/tasks.py', line 721, in async sleep()
  |   File '/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/ipykernel_6333/1577890246.py', line 2, in async child()
  + Awaited by:
    * Task(name='parent-task', id=0x10bbb3230)
      + Call stack:
      |   File '/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/ipykernel_6333/1577890246.py', line 12, in async parent()
      + Awaited by:
        * Task(name='Task-96', id=0x10bbb3e70)
          + Call stack:
          |   File '/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/ipykernel_6333/1577890246.py', line 14, in async <module>()
          |   File '/private/tmp/claude-501/-Users-gaurvendrasingh-Desktop

🆕 **Python 3.14** can inspect a **running** process from outside:
```bash
python -m asyncio ps 12345        # table of all pending tasks in process 12345
python -m asyncio pstree 12345    # tree: which task awaits which
```
(Needs the same permissions as the sampling profiler, Module 12 · 05.)

### ❌ Blocking the event loop
A single `time.sleep(2)`, CPU-heavy loop or synchronous `requests.get` freezes **all** tasks. Fix with `await asyncio.to_thread(func)`, `loop.run_in_executor(pool, func)`, async libraries, or processes for CPU work.

## 11. The async ecosystem

| Need | Library |
|---|---|
| HTTP client | **httpx** (`AsyncClient`), aiohttp |
| Web servers / APIs | **FastAPI** / Starlette (on uvicorn), aiohttp, Litestar |
| Databases | **asyncpg**, SQLAlchemy 2 asyncio, motor (MongoDB), redis-py asyncio |
| Files | aiofiles (or `to_thread`) |
| Structured concurrency on any backend | **anyio** (runs on asyncio or trio), **trio** |
| Testing | pytest-asyncio, anyio's pytest plugin (Module 07 · 04) |

## 📝 Summary
- The event loop runs one task at a time and switches at `await`; Tasks wrap coroutines; Futures hold results; anything with `__await__` is awaitable.
- Keep references to background tasks (or use `TaskGroup`); choose `TaskGroup`/`gather`/`as_completed`/`wait` deliberately.
- Cancellation: clean up in `finally`, re-raise `CancelledError`, `shield` critical work, `timeout`/`timeout_at`.
- Async locks/events/semaphores/barriers/queues prevent races between tasks; `contextvars` carry request context.
- Use debug mode, task names, call graphs and `python -m asyncio ps` to debug; never block the loop.

## 🏋️ Practice
1. Write a crawler that fetches pages with a `Semaphore(5)`, a `TaskGroup`, an overall `timeout`, and retries with exponential back-off.
2. Implement an async `RateLimiter` (N calls per second) with `asyncio.Lock` and `loop.time()`.
3. Add a `request_id` `ContextVar` to a logging `Filter` so every log line from a task includes its request ID.
4. Create a bug where a task is cancelled but swallows `CancelledError`; observe the problem, then fix it.